#### READ THE PRODUCTS DATA FROM VOLUME

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/Products/'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/products'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'PRODUCTS'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
products_schema = """product_id STRING, 
                     product_category_name STRING,
                     product_name_lenght STRING,
                     product_description_lenght STRING,
                     product_photos_qty STRING,
                     product_weight_g STRING,
                     product_length_cm STRING,
                     product_height_cm STRING,
                     product_width_cm STRING
                 """

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.PRODUCTS`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import builtins

def write_audit(run_id, status, start_time, rows=None, error=None):
    end = datetime.now()
    row = (run_id, "PRODUCTS_LOAD", "BRONZE", BASE_DIR, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
    (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

def current_version(table):
    try:
        return DeltaTable.forName(spark, table).history(1).select("version").first()[0]
    except Exception:          # table doesn't exist yet (first run)
        return -1

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
    version_before = current_version(TARGET)
    products_df  = (
                spark.readStream.format('cloudFiles')
                    .option('cloudFiles.format', 'csv')
                    .option('cloudFiles.schemaEvolutionMode', 'rescue')
                    .option('header', 'true')
                    .schema(products_schema)
                    .load(f'{BASE_DIR}')
            )
    products_trans_df = products_df.select(
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght", 
        "product_photos_qty", 
        "product_weight_g", 
        "product_length_cm", 
        "product_height_cm", 
        "product_width_cm",
        "_metadata.file_name",
        "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())


    qry = (
        products_trans_df.writeStream.format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_LOCATION)
        .trigger(availableNow=True)
        .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
    )
    qry.awaitTermination()

    rows_written = int(
        DeltaTable.forName(spark, TARGET).history()
            .filter((F.col("version") > version_before) &
                    (F.col("operation") == "STREAMING UPDATE"))
            .select(F.col("operationMetrics")["numOutputRows"].cast("long").alias("n"))
            .agg(F.coalesce(F.sum("n"), F.lit(0))).first()[0]
    )

except Exception:
    status, err = "FAILED", traceback.format_exc()
    raise
finally:
    write_audit(run_id, status, start_time, rows_written, err)